In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [17]:
df = pd.read_csv("btcusd_1-min_data.csv")
df.head()
df.describe()

,Timestamp,Open,High,Low,Close,Volume
count,7.502086e+06,7.502086e+06,7.502086e+06,7.502086e+06,7.502086e+06,7.502086e+06
mean,1.550480e+09,2.267265e+04,2.268070e+04,2.266443e+04,2.267267e+04,5.056240e+00
std,1.299483e+08,3.092683e+04,3.093526e+04,3.091834e+04,3.092685e+04,2.181390e+01
min,1.325412e+09,3.800000e+00,3.800000e+00,3.800000e+00,3.800000e+00,0.000000e+00
25%,1.437943e+09,4.510500e+02,4.512700e+02,4.509900e+02,4.510500e+02,2.110596e-02
50%,1.550475e+09,7.679455e+03,7.684370e+03,7.673360e+03,7.679320e+03,4.576846e-01
75%,1.663006e+09,3.582553e+04,3.584885e+04,3.580045e+04,3.582526e+04,2.859616e+00
max,1.775607e+09,1.262020e+05,1.262720e+05,1.261580e+05,1.262020e+05,5.853852e+03


In [19]:
df = df.info

AttributeError: 'function' object has no attribute 'info'

In [13]:
df = pd.read_csv("btcusd_1-min_data.csv")

# Timestamp di data ini format UNIX (detik), ubah ke datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')

# Data per menit terlalu banyak -> resample ke HARIAN (ambil Close harian)
df = df.set_index('Timestamp')
df_daily = df['Close'].resample('D').last().dropna().reset_index()

# Rename sesuai format Prophet
df_prophet = df_daily.rename(columns={'Timestamp': 'ds', 'Close': 'y'})

print(f"Total data harian: {len(df_prophet)} hari")
print(f"Dari: {df_prophet['ds'].min().date()} sampai {df_prophet['ds'].max().date()}")
df_prophet.head()

Total data harian: 5212 hari
Dari: 2012-01-01 sampai 2026-04-08


,ds,y
0,2012-01-01,4.84
1,2012-01-02,5.00
2,2012-01-03,5.29
3,2012-01-04,5.57
4,2012-01-05,6.42


In [20]:
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.1  # Sensitivitas terhadap perubahan tren
)

model.fit(df_prophet)
print("Model berhasil dilatih!")

21:17:02 - cmdstanpy - INFO - Chain [1] start processing
21:17:07 - cmdstanpy - INFO - Chain [1] done processing


Model berhasil dilatih!


In [23]:
# 5 tahun = 365 * 5 = 1825 hari
future = model.make_future_dataframe(periods=1825, freq='D')
forecast = model.predict(future)

print(f"Prediksi hingga: {forecast['ds'].max().date()}")
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10)

Prediksi hingga: 2031-04-07


,ds,yhat,yhat_lower,yhat_upper
7027,2031-03-29,254383.315917,86874.725371,411340.446682
7028,2031-03-30,254519.707940,88276.665713,409377.000625
7029,2031-03-31,254721.689766,90942.316498,411075.062923
7030,2031-04-01,254866.771437,90790.259198,410947.591812
7031,2031-04-02,255128.470902,88466.698475,413450.979614
7032,2031-04-03,255258.796825,88412.263278,414193.684110
7033,2031-04-04,255429.213556,91826.657932,407881.722153
7034,2031-04-05,255630.101176,90705.621825,408309.157125
7035,2031-04-06,255809.321077,87876.982822,411346.635958
7036,2031-04-07,256051.168599,89056.500987,416313.622947


In [25]:
# ============================================================
# CELL 5 - Evaluasi Akurasi (pada data yang sudah ada)
# ============================================================
# Gabungkan data aktual dengan prediksi
merged = df_prophet.merge(forecast[['ds', 'yhat']], on='ds')

mae  = mean_absolute_error(merged['y'], merged['yhat'])
mape = np.mean(np.abs((merged['y'] - merged['yhat']) / merged['y'])) * 100 

print(f"MAE  (rata-rata error dalam USD) : ${mae:,.2f}")
print(f"MAPE (rata-rata error dalam %)   : {mape:.2f}%")


MAE  (rata-rata error dalam USD) : $4,276.51
MAPE (rata-rata error dalam %)   : 989.51%


In [28]:
fig = go.Figure()

# Area ketidakpastian (pita biru muda)
fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat_upper'],
    line=dict(width=0), showlegend=False, hoverinfo="skip"
))
fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat_lower'],
    line=dict(width=0), fill='tonexty',
    fillcolor='rgba(0, 100, 255, 0.15)',
    name='Rentang Ketidakpastian'
))

# Garis prediksi
fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat'],
    mode='lines', name='Prediksi Prophet',
    line=dict(color='royalblue', width=2)
))

# Data aktual
fig.add_trace(go.Scatter(
    x=df_prophet['ds'], y=df_prophet['y'],
    mode='markers', name='Harga Aktual BTC',
    marker=dict(color='orange', size=1.5, opacity=0.4)
))

# Garis pemisah "sekarang vs masa depan"
last_date = df_prophet['ds'].max()
fig.add_vline(
    x=last_date.timestamp() * 1000,
    line_dash="dash",
    line_color="gray",
    annotation_text="← Aktual | Prediksi →",
    annotation_position="top"
)

fig.update_layout(
    title='Prediksi Harga Bitcoin (BTC/USD) - 5 Tahun ke Depan',
    xaxis_title='Tanggal',
    yaxis_title='Harga (USD)',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


In [29]:
future_only = forecast[forecast['ds'] > df_prophet['ds'].max()].copy()
future_only['tahun'] = future_only['ds'].dt.year

summary = future_only.groupby('tahun').agg(
    Prediksi_Rata2=('yhat', 'mean'),
    Prediksi_Min  =('yhat_lower', 'min'),
    Prediksi_Max  =('yhat_upper', 'max')
).round(2)

print("\n📊 Ringkasan Prediksi per Tahun:")
print(summary.to_string())


📊 Ringkasan Prediksi per Tahun:
       Prediksi_Rata2  Prediksi_Min  Prediksi_Max
tahun                                            
2026        123754.14     103545.57     147114.77
2027        148166.83     119017.35     195835.03
2028        176793.44     119536.01     257494.80
2029        205420.68     107865.34     321018.07
2030        234008.36      90959.78     394547.40
2031        251168.53      85968.33     416313.62
